In [ ]:
# LIBRERIAS DE TRABAJO
#-----------------------------------
import pandas as pd
import requests
import os
import io
import unicodedata #para manejo de tildes

# Para etapa de ingesta
#-------------------------------------- 
from sqlalchemy import create_engine, text
from sqlalchemy.types import String
import getpass #para manejo de password seguro en conexion a Mysql
import sqlparse #diseñada para entender la anatomía de un archivo SQL, ignorar comentarios y separar las sentencias correctamente


In [ ]:
# NAVEGA POR RELEASES de GITHUB PARA ENCONTRAR LOS ARCHIVOS EXISTENTES

user = "Marcelo-F-Martin"
repo = "PI_UA_pipeline_analisis_de_cobranza"

# URL de la API para la última release
api_url = f"https://api.github.com/repos/{user}/{repo}/releases/latest"

response = requests.get(api_url)
assets = response.json().get('assets', []) # [] es el segundo argumento del .get(). Significa: "Si por alguna razón la llave 'assets' no existe, no rompas el programa, devuélveme una lista vacía".
assets


## Inicia proceso ETL
#### 1- Extracción:
#####   Obtención de datos crudos desde la fuente y almacenamiento en Dataframe de Pandas

In [ ]:
df_definitivo_por_libro = []   
df_final = []

#====== INICIO 2º bucle: recorre hojas del archivo ===============================

for asset in assets:
    
    # Filtramos para que solo lea los archivos .xls
    if asset['name'].endswith('.xls'):
        print(f"Detectado archivo: {asset['name']}")
        
        # El campo 'browser_download_url' es el link directo
        
    hojas_consolidadas_por_libro = [] # se crea una lista para incorporar los df por cada hoja de cada libro, para luego agrupar todos.
    
    file_url = asset['browser_download_url']
    file_data = requests.get(file_url).content
    
    ## df_temp = pd.read_excel(io.BytesIO(file_data), engine='xlrd') -----se incorpora esta linea en la linea siguiente.

    hojas_brutas = pd.read_excel(io.BytesIO(file_data), engine='xlrd', sheet_name=None, header=None)

#====== INICIO 1º bucle: recorre hojas del archivo ===============================
        
    for nombre_hoja, df_bruto in hojas_brutas.items():
                   
        # Aquí 'df_bruto' es el DataFrame bruto de la hoja actual
        print(f"✅ Hoja '{nombre_hoja}' cargada correctamente. Filas brutas: {len(df_bruto)}")
        # Retornamos el diccionario para el siguiente paso del procesamiento

        print(f"  --- Buscando clave en Hoja '{nombre_hoja}' ---")
        
        encabezado_clave = "Periodo"
        indice_encabezado_fila = df_bruto[df_bruto.apply(lambda row: encabezado_clave in row.astype(str).values, axis=1)].index
        
        fila_inicio = indice_encabezado_fila[0]
        print(f"✅ Encabezado clave '{encabezado_clave}' encontrado en la Fila (índice 0): {fila_inicio}")

        fila_encabezado = df_bruto.iloc[fila_inicio]

        # 2b. 'col_inicio' será el índice de la columna donde se encuentra la clave
        # Esto es crucial para ignorar las columnas añadidas a la izquierda.
        
        col_inicio = fila_encabezado[fila_encabezado.astype(str) == encabezado_clave].index[0]
        print(f"  ✅ Columna de inicio de datos (índice 0): {col_inicio}")

        # Por eficiencia, si ya tienes el df_bruto, lo mejor es usarlo para el recorte:
        df_bruto.columns = df_bruto.iloc[fila_inicio] # Asignar la fila encontrada como nuevo encabezado
        df_limpio = df_bruto[fila_inicio + 1:].reset_index(drop=True) # Eliminar filas superiores

        df_final_hoja = df_limpio.copy()
        df_final_hoja['hoja_origen'] = nombre_hoja # se inserta nueva columna que identifica la Hoja de Origen

        if not df_final_hoja.empty:
            hojas_consolidadas_por_libro.append(df_final_hoja)
            print(f"  ✅ Hoja '{nombre_hoja}' añadida a la consolidación. Registros: {len(df_final_hoja)}")
        else:
            print(f"  Advertencia: Hoja '{nombre_hoja}' vacía después de la limpieza. No se añadió.")


       
    df_archivo_unificado = pd.concat(hojas_consolidadas_por_libro, ignore_index=True)
    df_definitivo_por_libro.append(df_archivo_unificado)
            
#====== FIN 1º bucle: recorre hojas del archivo ===============================
#====== FIN 2º bucle: recorre hojas del archivo ===============================

# Unificación final
df_final = pd.concat(df_definitivo_por_libro, ignore_index=True)


In [ ]:
df_final.head()

#### 2- Exploración y Transformación

In [ ]:
df_final.info()

In [ ]:
def limpiar_tilde(texto):
    # 1. Normaliza el texto a la forma NFD (Descomposición)
    # Esto separa la 'á' en 'a' + 'tilde combinable'
    texto_normalizado = unicodedata.normalize('NFD', texto)
    
    # 2. Filtra y mantiene solo los caracteres que no sean "marcas" de acento (Mn)
    texto_sin_acentos = "".join(
        c for c in texto_normalizado if unicodedata.category(c) != 'Mn'
    )
    
    return texto_sin_acentos

In [ ]:
# 1_estandariza a minusculas
# 2_elimina espacios en blanco
# 3_reemplaza vacíos por '_'
# 4_aplica función definida para limpiar tildes
#________________________________________________

df_final.columns = df_final.columns.str.lower().str.strip().str.replace(' ','_')
df_final.columns = [limpiar_tilde(col) for col in df_final.columns]
df_final.columns

In [ ]:
# seleccion de columnas relevantes a importar a la Base de Datos
lista_columnas_seleccionadas = ['periodo','asesor','fecha_operacion','contrato','comision', 'valor_neto', 'porcentaje_comision','forma_pago', 'numero_recibo'  ]
df_final = df_final[lista_columnas_seleccionadas]

In [ ]:
df_final.head()

In [ ]:
# modifica tipo de datos. se comenta provisoriamente para evaluar necesidad de cambio de tipo de datos.

#df_final = df_final.astype({'comision': float, 'valor_neto': float, 'porcentaje_comision': float})
df_final.info()

In [ ]:
#round(df_final.describe(),2)   se comenta esta linea porque los dtype son todos object y da un warning

In [ ]:
# Verifica existencia de PK
df_final.nunique()

In [ ]:
# Esta celda puede ser opcional para exportar el dataframe en CSV.

#ruta_archivo = r"C:\Users\orglo\OneDrive\Desktop\marcelo\Proyectos\PI"
#nombre_csv_salida = f"comisiones_consolidadas.csv"
#ruta_salida_completa = os.path.join(ruta_archivo, nombre_csv_salida)
#df_final.to_csv(ruta_salida_completa, index=False, encoding='utf-8')


In [ ]:
# se limita a 10 registros para prueba de ingestion en mysql

#df_final = df_final.head(100)
#df_final

#### SQL - ejecuta scripts sql: 
##### 1- crea base de datos y tablas
##### 2- inserta datos en tabla temporal
##### 3- inserta datos en tabla fact
##### 4- crea vistas en BD capa_dos

In [ ]:
# Verifique los datos de su entorno en MySQL
usuario = "root"
host = "localhost"
puerto = "3306"
bd_capa_uno = "cobranzas_capa_uno"
bd_capa_dos = "cobranzas_capa_dos"
tabla_temp = "temp_cobranzas"
df = df_final

url_sql_ddl = 'https://raw.githubusercontent.com/Marcelo-F-Martin/PI_UA_pipeline_analisis_de_cobranza/refs/heads/main/SQL/PI_UA_DDL_proyecto_03%2Btablas_dim%2Bcapa_dos.sql'
url_sql_insert_fact = 'https://raw.githubusercontent.com/Marcelo-F-Martin/PI_UA_pipeline_analisis_de_cobranza/refs/heads/main/SQL/PI_UA_DDL_proyecto_insert_en_fact.sql'
url_sql_vistas = 'https://raw.githubusercontent.com/Marcelo-F-Martin/PI_UA_pipeline_analisis_de_cobranza/refs/heads/main/SQL/PI_UA_capa_dos_vistas.sql'

try:
    # Descargar el contenido del archivo
    respuesta_1 = requests.get(url_sql_ddl)
    respuesta_2 = requests.get(url_sql_insert_fact)    
    respuesta_3 = requests.get(url_sql_vistas)    
    
    if respuesta_1.status_code == 200 and respuesta_2.status_code == 200 and respuesta_3.status_code == 200:
        script_sql_1 = respuesta_1.text
        print("✅ Script SQL 1 descargado correctamente desde GitHub.")

        script_sql_2 = respuesta_2.text
        print("✅ Script SQL 2 descargado correctamente desde GitHub.")

        script_sql_3 = respuesta_3.text
        print("✅ Script SQL 3 descargado correctamente desde GitHub.")

        
        # Conectar Ejecutar en MySQL
        
        password = getpass.getpass("Ingrese su contraseña de MySQL: ")    
        engine_create = create_engine(f"mysql+pymysql://{usuario}:{password}@{host}:{puerto}") # crea una nueva bd
        engine_insert = create_engine(f"mysql+pymysql://{usuario}:{password}@{host}:{puerto}/{bd_capa_uno}") # inserta datos en temp_cobranzas
        engine_vista = create_engine(f"mysql+pymysql://{usuario}:{password}@{host}:{puerto}/{bd_capa_dos}") # crea vistas

        # 1.DDL crea base de datos y tablas        
        with engine_create.begin() as connection:
            
            # Dividimos por punto y coma para ejecutar sentencias individuales
            for statement in script_sql_1.split(';'):
                if statement.strip():
                    connection.execute(text(statement))

        # 2.Ingesta el dataframe en tabla temporal (truncate + insert)
        with engine_insert.begin() as connection:
            connection.execute(text(f"TRUNCATE TABLE {tabla_temp}"))
            print(f"🧹 Tabla '{tabla_temp}' limpiada (registros eliminados).")
            
            df.to_sql(
                name=tabla_temp,
                con=connection,
                if_exists='append', 
                index=False,
                chunksize=1000, 
                method='multi' # Para que sea masivo y no registro por registro
            )
           
        print(f"✅ Ingesta exitosa: {len(df)} registros insertados.")

        # 3.Pasa datos de tabla_temp a tabla_fact        
        with engine_insert.begin() as connection:
             
            # Dividimos por punto y coma para ejecutar sentencias individuales
            for statement in script_sql_2.split(';'):
                if statement.strip():
                    connection.execute(text(statement))
        
        print("🚀 Ejecución en MySQL finalizada exitosamente.")

        # 4.Crea vistas en BD cobranzas_capa_dos        
        with engine_vista.begin() as connection:
             
            # Dividimos por punto y coma para ejecutar sentencias individuales
            for statement in script_sql_3.split(';'):
                if statement.strip():
                    connection.execute(text(statement))
        
        print("🚀 Ejecución en MySQL finalizada exitosamente.")


        
    else:
        print(f"❌ Error al acceder al archivo Nº 1: Estado {respuesta_1.status_code}")
        print(f"❌ Error al acceder al archivo Nº 2: Estado {respuesta_2.status_code}")
        print(f"❌ Error al acceder al archivo Nº 3: Estado {respuesta_3.status_code}")

except Exception as e:
    print(f"❌ Error durante el proceso: {e}")

print(f"===================================================")
print(f"===================================================")